Import Libraries

In [ ]:
# %pip install yfinance -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# import yfinance as yf

# print("yfinance installed successfully!")

yfinance installed successfully!


In [3]:
import pandas as pd
import numpy as np
import yfinance as yf
import joblib

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.model_selection import GridSearchCV

import warnings
warnings.filterwarnings("ignore")

Create Folders

In [4]:
import os

os.makedirs("models", exist_ok=True)
os.makedirs("metrics", exist_ok=True)

Stocks List

In [5]:
stocks = [
    "TSLA",
    "AAPL",
    "MSFT"
]

Feature Engineering Function

In [6]:
def create_features(df):

    df['Lag_1'] = df['Close'].shift(1)
    df['Lag_2'] = df['Close'].shift(2)
    df['Lag_3'] = df['Close'].shift(3)
    df['Lag_5'] = df['Close'].shift(5)

    df['MA_5'] = df['Close'].rolling(5).mean()
    df['MA_10'] = df['Close'].rolling(10).mean()
    df['MA_20'] = df['Close'].rolling(20).mean()

    df['Volatility'] = df['Close'].rolling(10).std()

    df['Return'] = df['Close'].pct_change()

    # Next day closing price
    df['Target'] = df['Close'].shift(-1)

    df.dropna(inplace=True)

    return df

Evaluation Function

In [7]:
def evaluate(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)

    mse = mean_squared_error(y_true, y_pred)

    rmse = np.sqrt(mse)

    r2 = r2_score(y_true, y_pred)

    return mae, mse, rmse, r2

Train All Stocks

In [8]:
results = []

for stock in stocks:

    print(f"\nProcessing {stock}...")

    # ====================================
    # Download Data
    # ====================================

    df = yf.download(
        stock,
        start="2018-01-01",
        end="2025-01-01",
        auto_adjust=True
    )

    df = create_features(df)

    # ====================================
    # Features
    # ====================================

    feature_cols = [
        'Lag_1',
        'Lag_2',
        'Lag_3',
        'Lag_5',
        'MA_5',
        'MA_10',
        'MA_20',
        'Volatility',
        'Return',
        'Volume'
    ]

    X = df[feature_cols]

    y = df['Target']

    # ====================================
    # Time Based Split
    # ====================================

    split = int(len(df) * 0.8)

    X_train = X[:split]
    X_test = X[split:]

    y_train = y[:split]
    y_test = y[split:]

    # ====================================
    # Linear Regression
    # ====================================

    lr = LinearRegression()

    lr.fit(X_train, y_train)

    lr_pred = lr.predict(X_test)

    lr_metrics = evaluate(y_test, lr_pred)

    # ====================================
    # Decision Tree
    # ====================================

    dt = DecisionTreeRegressor(
        max_depth=8,
        random_state=42
    )

    dt.fit(X_train, y_train)

    dt_pred = dt.predict(X_test)

    dt_metrics = evaluate(y_test, dt_pred)

    # ====================================
    # Random Forest
    # ====================================

    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        random_state=42,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    rf_pred = rf.predict(X_test)

    rf_metrics = evaluate(y_test, rf_pred)

    # ====================================
    # Save Random Forest
    # ====================================

    joblib.dump(
        rf,
        f"models/{stock}_model.pkl"
    )

    # ====================================
    # Results
    # ====================================

    results.append([
        stock,
        "Linear Regression",
        *lr_metrics
    ])

    results.append([
        stock,
        "Decision Tree",
        *dt_metrics
    ])

    results.append([
        stock,
        "Random Forest",
        *rf_metrics
    ])

    print(f"{stock} completed")


Processing TSLA...


[*********************100%***********************]  1 of 1 completed


TSLA completed

Processing AAPL...


[*********************100%***********************]  1 of 1 completed


AAPL completed

Processing MSFT...


[*********************100%***********************]  1 of 1 completed


MSFT completed


Create Metrics Table

In [9]:
metrics_df = pd.DataFrame(
    results,
    columns=[
        "Stock",
        "Model",
        "MAE",
        "MSE",
        "RMSE",
        "R2"
    ]
)

metrics_df

,Stock,Model,MAE,MSE,RMSE,R2
0,TSLA,Linear Regression,7.134113,105.162617,10.254883,0.971850
1,TSLA,Decision Tree,11.347618,333.687548,18.267117,0.910680
2,TSLA,Random Forest,9.767262,290.663192,17.048847,0.922196
3,AAPL,Linear Regression,2.323430,9.270799,3.044799,0.985005
4,AAPL,Decision Tree,17.345646,638.859526,25.275671,-0.033351
5,AAPL,Random Forest,17.966467,715.992914,26.758044,-0.158114
6,MSFT,Linear Regression,4.167799,29.566527,5.437511,0.980689
7,MSFT,Decision Tree,70.498512,6365.673025,79.785168,-3.157574
8,MSFT,Random Forest,59.917580,4687.858114,68.467935,-2.061753


In [10]:
metrics_df.to_csv(
    "metrics/metrics.csv",
    index=False
)

print("Metrics Saved")

Metrics Saved


In [ ]:
metrics_df.sort_values(
    by="R2",
    ascending=False
)

,Stock,Model,MAE,MSE,RMSE,R2
3,AAPL,Linear Regression,2.323430,9.270799,3.044799,0.985005
6,MSFT,Linear Regression,4.167799,29.566527,5.437511,0.980689
0,TSLA,Linear Regression,7.134113,105.162617,10.254883,0.971850
2,TSLA,Random Forest,9.767262,290.663192,17.048847,0.922196
1,TSLA,Decision Tree,11.347618,333.687548,18.267117,0.910680
4,AAPL,Decision Tree,17.345646,638.859526,25.275671,-0.033351
5,AAPL,Random Forest,17.966467,715.992914,26.758044,-0.158114
8,MSFT,Random Forest,59.917580,4687.858114,68.467935,-2.061753
7,MSFT,Decision Tree,70.498512,6365.673025,79.785168,-3.157574


: 